# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adithyajupally/flyrank-ml-internship-jupally/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

This project asks whether we can use search and page performance data to find pages that may be getting fewer clicks than expected.

The goal is to help an SEO or content team decide which pages should be checked first.

The model will give each page an opportunity score and rank the pages from higher to lower priority.

This score is only a decision-support signal. It does not mean that a page definitely needs to be changed.

In [1]:
!git clone https://github.com/adithyajupally/flyrank-ml-internship-jupally.git
%cd flyrank-ml-internship-jupally

fatal: destination path 'flyrank-ml-internship-jupally' already exists and is not an empty directory.
/content/flyrank-ml-internship-jupally


In [2]:
research_question = ("Can search and page performance data help find pages that may be getting fewer clicks than expected?")

decision = ("Prioritize pages for SEO and content review based on their estimated CTR opportunity.")

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision)

Research question:
Can search and page performance data help find pages that may be getting fewer clicks than expected?

Decision supported:
Prioritize pages for SEO and content review based on their estimated CTR opportunity.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

I used the `content_refresh_anonymized.csv` dataset.

The dataset has 30,000 rows and 44 columns. Each row represents one content page for one client.

The dataset does not have daily dates, so the February and March time periods cannot be checked directly.

The target for this project is `ctr`.

I will use search, content, traffic, and engagement signals as features. IDs such as `content_id` and `client_id` will only be used as context and not as model features.

I will exclude `clicks_90d`, `provider_used`, and `model_used` from the model.

The results should be treated as a useful signal for deciding which pages to review. They should not be treated as proof that changing a page will increase its CTR.

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Dataset shape:", df.shape)

print("\nNumber of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nDuplicate content IDs:", df["content_id"].duplicated().sum())

print("\nTarget column: ctr")

print("\nColumns:")
print(df.columns.tolist())

missing_values = df.isna().sum().sort_values(ascending=False)
missing_values = missing_values[missing_values > 0]

print("\nColumns with missing values:")
print(missing_values)

Dataset shape: (30000, 44)

Number of rows: 30000
Number of columns: 44

Duplicate content IDs: 0

Target column: ctr

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Columns with missing values:
provider_used        21438
word_count            7699
char_count       

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

I will use `ctr` as the target because it shows how often a page gets clicks from its impressions.

I will use search, content, traffic, and engagement columns as features.

I will not use IDs or click-related columns as model features because they can give the model information that is too closely related to the target.

I will use a tree-based regression model because CTR is a number.

I will split the data into training and test sets and use the same test data to compare the model with a simple baseline.

I will also check for possible data leakage before training the model.

In [4]:
# Target
target = "ctr"

# Columns that should not be used as features
excluded_columns = ["content_id","client_id","clicks_90d","clicks_last_30d","clicks_prev_30d","provider_used","model_used"]

# Features
feature_columns = [col for col in df.columns if col not in excluded_columns + [target]]

# Possible leakage
possible_leakage = [col for col in feature_columns if "click" in col.lower() or "ctr" in col.lower()]

print("Possible leakage columns still in the features:", possible_leakage)

print("Target:", target)
print("Number of features:", len(feature_columns))
print("Features:")
print(feature_columns)

Possible leakage columns still in the features: []
Target: ctr
Number of features: 36
Features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

I will compare the Random Forest model with a simple baseline that predicts the average CTR from the training data.

Both methods are tested on the same test data.

Lower MAE and RMSE are better. A higher R² means that the model explains more of the variation in CTR.

The results show how well the model estimates CTR. They do not prove that changing a page will increase its CTR.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Remove rows where the target is missing
model_df = df.dropna(subset=["ctr"]).copy()

X = model_df[feature_columns]
y = model_df["ctr"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

# Separate numeric and categorical features
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Numeric features: 26
Categorical features: 10
Training rows: 24000
Test rows: 6000


In [6]:
# Fill missing numeric values
numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])

# Fill missing text values and convert text to numbers
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Tree-based regression model
model = RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1)

model_pipeline = Pipeline([("preprocessor", preprocessor),("model", model)])

model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)

print("Model trained successfully.")

Model trained successfully.


In [7]:
# Simple baseline: predict the average training CTR for every test row
baseline_prediction = y_train.mean()

y_baseline = [baseline_prediction] * len(y_test)

print("Baseline CTR prediction:", round(baseline_prediction, 4))

Baseline CTR prediction: 0.5095


In [8]:
import numpy as np

# ML model metrics
model_mae = mean_absolute_error(y_test, y_pred)
model_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
model_r2 = r2_score(y_test, y_pred)

# Baseline metrics
baseline_mae = mean_absolute_error(y_test, y_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_baseline))
baseline_r2 = r2_score(y_test, y_baseline)

# Results table
results = pd.DataFrame({
    "Model": ["Baseline", "Random Forest"],
    "MAE": [baseline_mae, model_mae],
    "RMSE": [baseline_rmse, model_rmse],
    "R2": [baseline_r2, model_r2]
})
results

,Model,MAE,RMSE,R2
0,Baseline,0.715174,3.224232,-0.000004
1,Random Forest,0.568720,2.876552,0.204036


The Random Forest model performed better than the simple baseline on the same test set.

It had lower MAE and RMSE and a higher R².

The model can be used as a useful signal for finding pages that may need review.

## 5. Limitations

*What this work cannot claim.*

This project has some limitations.

- The dataset does not have daily dates, so we cannot check dates directly.
- We cannot say that one factor directly causes a higher or lower CTR.
- A high opportunity score does not mean that a page definitely needs a change.
- The model is mainly useful for finding pages that should be reviewed first.
- The results may not work the same way on a different dataset.

In [9]:
print("Main limitations:")
print("1. The dataset does not provide daily dates.")
print("2. The model shows patterns, not direct causes.")
print("3. High opportunity does not guarantee that a page needs a change.")
print("4. Results may differ on other datasets.")

Main limitations:
1. The dataset does not provide daily dates.
2. The model shows patterns, not direct causes.
3. High opportunity does not guarantee that a page needs a change.
4. Results may differ on other datasets.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The pages with the highest opportunity scores should be reviewed first.

The reason code shows why each page was selected.

These scores are only for prioritizing pages for review.

In [10]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Min-max normalization
position_score = (df["avg_position"].max() - df["avg_position"]) / (df["avg_position"].max() - df["avg_position"].min())

impression_score = (df["impressions_90d"] - df["impressions_90d"].min()) / (df["impressions_90d"].max() - df["impressions_90d"].min())

low_ctr_score = 1 - df["ctr"]

# Baseline opportunity score
df["opportunity_score"] = (0.4 * position_score+ 0.4 * low_ctr_score+ 0.2 * impression_score)

# Reason codes
df["reason_code"] = "LOW_CTR_FOR_POSITION"

df.loc[(df["avg_position"] <= 20) & (df["ctr"] < 0.20),"reason_code"] = "GOOD_POSITION_LOW_CTR"

df.loc[(df["impressions_90d"] > df["impressions_90d"].median()) & (df["ctr"] < 0.20),"reason_code"] = "HIGH_IMPRESSIONS_LOW_CTR"

# Top 20 pages
recommendations = (df[["content_id","avg_position","ctr","impressions_90d","opportunity_score","reason_code"]]
                    .sort_values("opportunity_score", ascending=False).head(20))

recommendations.head(10)

,content_id,avg_position,ctr,impressions_90d,opportunity_score,reason_code
6653,content_5fe46e04994d,4.2,0.14,517715,0.937143,HIGH_IMPRESSIONS_LOW_CTR
26844,content_8c19996aa890,2.5,0.15,509252,0.932649,HIGH_IMPRESSIONS_LOW_CTR
19636,content_2cb567c3c89b,22.2,0.10,497727,0.916033,HIGH_IMPRESSIONS_LOW_CTR
17812,content_aaef01a50def,5.4,0.25,517109,0.890950,LOW_CTR_FOR_POSITION
7678,content_8451fc6f034d,2.3,0.03,272144,0.889377,HIGH_IMPRESSIONS_LOW_CTR
3394,content_36ff89c8214e,7.3,0.05,295097,0.882081,HIGH_IMPRESSIONS_LOW_CTR
7445,content_c8e9d6ab9013,9.7,0.00,208678,0.864778,HIGH_IMPRESSIONS_LOW_CTR
29879,content_1a9e894be2e2,4.0,0.23,416180,0.862245,LOW_CTR_FOR_POSITION
6903,content_c84a0ab98e90,7.8,0.03,223271,0.861518,HIGH_IMPRESSIONS_LOW_CTR
26531,content_cb112fce36be,5.6,0.16,309910,0.846579,HIGH_IMPRESSIONS_LOW_CTR


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*


The main artifact is the ranked recommendation table.

I will also show a small chart of the top pages to make the results easier to understand.

In [11]:
import plotly.express as px

fig = px.bar(
    recommendations.head(10),
    x="content_id",
    y="opportunity_score",
    hover_data=["ctr", "avg_position", "impressions_90d", "reason_code"],
    title="Top 10 Pages by Opportunity Score"
)

fig.show()

In [12]:
fig = px.scatter(
    recommendations,
    x="avg_position",
    y="ctr",
    size="impressions_90d",
    hover_data=["content_id", "opportunity_score", "reason_code"],
    title="CTR vs Search Position"
)

fig.show()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
